# Stock Price Prediction using LSTM and Random Forest

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from datetime import datetime, timedelta
from abc import ABC, abstractmethod
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from keras.models import Sequential
from keras.layers import Dense, LSTM
from keras.callbacks import EarlyStopping
import warnings

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.style.use("fivethirtyeight")
%matplotlib inline

## Configuration

In [ ]:
TECH_LIST           = ["AAPL", "GOOG", "MSFT", "AMZN"]
COMPANY_NAMES       = ["APPLE", "GOOGLE", "MICROSOFT", "AMAZON"]
PRIMARY_TICKER      = "AAPL"
PRIMARY_TICKER_NAME = "Apple"

EDA_END_DATE    = datetime.now()
EDA_START_DATE = EDA_END_DATE - timedelta(days=365)
LSTM_START_DATE = "2012-01-01"
LSTM_END_DATE   = datetime.now()

LOOKBACK_WINDOW    = 60
TRAIN_SPLIT_RATIO  = 0.85
MOVING_AVG_WINDOWS = [10, 20, 50]

LSTM_UNITS_LAYER_1     = 128
LSTM_UNITS_LAYER_2     = 64
DENSE_UNITS            = 25
EPOCHS                 = 50
BATCH_SIZE             = 32
DROPOUT_RATE           = 0.2
RECURRENT_DROPOUT_RATE = 0.2
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MONITOR  = "val_loss"
VALIDATION_SPLIT        = 0.1
OPTIMIZER               = "adam"
LOSS                    = "mean_squared_error"

RF_N_ESTIMATORS = 100
RF_RANDOM_STATE = 42

NUM_FUTURE_DAYS = 60

## Model Definitions

In [ ]:
class BaseModel(ABC):

    @abstractmethod
    def train(self, x_train, y_train, **kwargs): ...

    @abstractmethod
    def predict(self, x, **kwargs): ...

    def evaluate(self, predictions, actuals):
        rmse = float(np.sqrt(np.mean((predictions - actuals) ** 2)))
        return {"rmse": rmse}

In [ ]:
class LSTMModel(BaseModel):

    def __init__(self):
        self._model  = self._build()
        self.history = None

    def _build(self):
        model = Sequential([
            LSTM(LSTM_UNITS_LAYER_1, return_sequences=True,
                 input_shape=(LOOKBACK_WINDOW, 1),
                 dropout=DROPOUT_RATE, recurrent_dropout=RECURRENT_DROPOUT_RATE),
            LSTM(LSTM_UNITS_LAYER_2, return_sequences=False,
                 dropout=DROPOUT_RATE, recurrent_dropout=RECURRENT_DROPOUT_RATE),
            Dense(DENSE_UNITS),
            Dense(1),
        ])
        model.compile(optimizer=OPTIMIZER, loss=LOSS)
        model.summary()
        return model

    def train(self, x_train, y_train, **kwargs):
        epochs     = kwargs.pop("epochs",     EPOCHS)
        batch_size = kwargs.pop("batch_size", BATCH_SIZE)
        early_stop = EarlyStopping(monitor=EARLY_STOPPING_MONITOR,
                                   patience=EARLY_STOPPING_PATIENCE,
                                   restore_best_weights=True, verbose=1)
        user_cb = kwargs.pop("callbacks", [])
        self.history = self._model.fit(
            x_train, y_train,
            epochs=epochs, batch_size=batch_size,
            validation_split=VALIDATION_SPLIT,
            callbacks=[early_stop] + list(user_cb), **kwargs)

    def predict(self, x, **kwargs):
        return self._model.predict(x, **kwargs)

In [ ]:
class RFModel(BaseModel):

    def __init__(self):
        self._model = RandomForestRegressor(
            n_estimators=RF_N_ESTIMATORS, random_state=RF_RANDOM_STATE, n_jobs=-1)

    @staticmethod
    def _flatten(x):
        return x.reshape(x.shape[0], -1) if x.ndim == 3 else x

    def train(self, x_train, y_train, **kwargs):
        self._model.fit(self._flatten(x_train), y_train)

    def predict(self, x, **kwargs):
        return self._model.predict(self._flatten(x)).reshape(-1, 1)

    @property
    def feature_importances(self):
        return self._model.feature_importances_

## Data Pipeline

In [ ]:
def download_eda_data():
    company_list = []
    for ticker, name in zip(TECH_LIST, COMPANY_NAMES):
        df = yf.download(ticker, EDA_START_DATE, EDA_END_DATE, progress=False)
        df["company_name"] = name
        for w in MOVING_AVG_WINDOWS:
            df[f"MA for {w} days"] = df["Close"].rolling(w).mean()
        df["Daily Return"] = df["Close"].pct_change()
        company_list.append(df)
    raw = yf.download(TECH_LIST, start=EDA_START_DATE, end=EDA_END_DATE, progress=False)
    return company_list, raw["Close"]

In [ ]:
def build_lstm_datasets(df):
    dataset           = df[["Close"]].values
    training_data_len = int(np.ceil(len(dataset) * TRAIN_SPLIT_RATIO))

    # Fit scaler only on training data to prevent data leakage
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaler.fit(dataset[:training_data_len])
    scaled_data = scaler.transform(dataset)

    train_data = scaled_data[:training_data_len]
    x_train, y_train = [], []
    for i in range(LOOKBACK_WINDOW, len(train_data)):
        x_train.append(train_data[i - LOOKBACK_WINDOW:i, 0])
        y_train.append(train_data[i, 0])
    x_train = np.array(x_train).reshape(-1, LOOKBACK_WINDOW, 1)
    y_train = np.array(y_train)

    test_data = scaled_data[training_data_len - LOOKBACK_WINDOW:]
    x_test = []
    y_test = dataset[training_data_len:]
    for i in range(LOOKBACK_WINDOW, len(test_data)):
        x_test.append(test_data[i - LOOKBACK_WINDOW:i, 0])
    x_test = np.array(x_test).reshape(-1, LOOKBACK_WINDOW, 1)

    return x_train, y_train, x_test, y_test, scaler, training_data_len, scaled_data

In [ ]:
def build_future_sequence(scaled_data, model, scaler):
    current_seq  = scaled_data[-LOOKBACK_WINDOW:].copy()
    preds_scaled = []
    for _ in range(NUM_FUTURE_DAYS):
        seq  = current_seq.reshape(1, LOOKBACK_WINDOW, 1)
        pred = model.predict(seq, verbose=0)
        preds_scaled.append(pred[0][0])
        current_seq = np.append(current_seq[1:], pred, axis=0)
    future_usd   = scaler.inverse_transform(np.array(preds_scaled).reshape(-1, 1))
    future_dates = pd.bdate_range(start=df.index[-1] + timedelta(days=1),
                                  periods=NUM_FUTURE_DAYS)
    return future_usd, future_dates

## Visualisation Functions

In [ ]:
def plot_closing_prices(company_list):
    fig, _ = plt.subplots(2, 2, figsize=(15, 10))
    for i, (company, ticker) in enumerate(zip(company_list, TECH_LIST), 1):
        ax = fig.add_subplot(2, 2, i)
        company["Close"].plot(ax=ax)
        ax.set_title(f"Closing Price of {ticker}")
    plt.tight_layout()
    plt.show()

def plot_volumes(company_list):
    fig, _ = plt.subplots(2, 2, figsize=(15, 10))
    for i, (company, ticker) in enumerate(zip(company_list, TECH_LIST), 1):
        ax = fig.add_subplot(2, 2, i)
        company["Volume"].plot(ax=ax)
        ax.set_title(f"Sales Volume for {ticker}")
    plt.tight_layout()
    plt.show()

def plot_moving_averages(company_list):
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    ma_cols = ["Close"] + [f"MA for {w} days" for w in MOVING_AVG_WINDOWS]
    for ax, company, name in zip(axes.flatten(), company_list, COMPANY_NAMES):
        company[ma_cols].plot(ax=ax)
        ax.set_title(name)
    fig.tight_layout()
    plt.show()

def plot_daily_returns(company_list):
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    for ax, company, name in zip(axes.flatten(), company_list, COMPANY_NAMES):
        company["Daily Return"].plot(ax=ax, legend=True, linestyle="--", marker="o")
        ax.set_title(name)
    fig.tight_layout()
    plt.show()

def plot_correlation_heatmaps(tech_rets, closing_df):
    plt.figure(figsize=(12, 10))
    plt.subplot(2, 2, 1)
    sns.heatmap(tech_rets.corr(), annot=True, cmap="summer")
    plt.title("Correlation of stock return")
    plt.subplot(2, 2, 2)
    sns.heatmap(closing_df.corr(), annot=True, cmap="summer")
    plt.title("Correlation of stock closing price")
    plt.tight_layout()
    plt.show()

def plot_close_history(df):
    plt.figure(figsize=(16, 6))
    plt.title(f"{PRIMARY_TICKER_NAME} Close Price History")
    plt.plot(df["Close"])
    plt.xlabel("Date", fontsize=18)
    plt.ylabel("Close Price USD ($)", fontsize=18)
    plt.show()

def plot_predictions(train, valid, future_df=None):
    plt.figure(figsize=(16, 8))
    plt.title(f"{PRIMARY_TICKER_NAME} Stock Price Prediction — LSTM vs Random Forest")
    plt.xlabel("Date", fontsize=18)
    plt.ylabel("Close Price USD ($)", fontsize=18)
    plt.plot(train["Close"],  color="blue",   label="Training Data")
    plt.plot(valid["Close"],  color="green",  label="Validation Actual")
    if "Predictions"    in valid.columns:
        plt.plot(valid["Predictions"],    color="red",    linewidth=1.5, label="LSTM Predictions")
    if "RF Predictions" in valid.columns:
        plt.plot(valid["RF Predictions"], color="orange", linewidth=1.5,
                 linestyle="--", label="RF Predictions")
    if future_df is not None:
        plt.plot(future_df["Future Predictions"], color="purple",
                 linestyle="--", label=f"Future {NUM_FUTURE_DAYS} Days")
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.show()

## Exploratory Data Analysis

In [ ]:
company_list, closing_df = download_eda_data()
tech_rets = closing_df.pct_change()

In [ ]:
# @title
plot_closing_prices(company_list)

In [ ]:
plot_volumes(company_list)

In [ ]:
plot_moving_averages(company_list)

In [ ]:
plot_daily_returns(company_list)

In [ ]:
sns.pairplot(tech_rets.dropna(), kind="reg")
plt.show()

In [ ]:
plot_correlation_heatmaps(tech_rets, closing_df)

## Data Preparation

In [ ]:
df = yf.download(PRIMARY_TICKER, start=LSTM_START_DATE, end=LSTM_END_DATE, progress=False)
print(df.shape)
df.tail()

In [ ]:
plot_close_history(df)

In [ ]:
x_train, y_train, x_test, y_test, scaler, train_len, scaled_data = build_lstm_datasets(df)

print(f"x_train : {x_train.shape}")
print(f"x_test  : {x_test.shape}")
print(f"Train rows: {train_len}  |  Test rows: {len(df) - train_len}")

## Training

### LSTM

In [ ]:
lstm = LSTMModel()
lstm.train(x_train, y_train)

In [ ]:
hist = lstm.history.history
plt.figure(figsize=(12, 4))
plt.plot(hist["loss"],     label="Train Loss")
plt.plot(hist["val_loss"], label="Val Loss", linestyle="--")
plt.title("LSTM Training Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.grid(True)
plt.show()

### Random Forest

In [ ]:
rf = RFModel()
rf.train(x_train, y_train)

## Evaluation

In [ ]:
preds_lstm = scaler.inverse_transform(lstm.predict(x_test))
preds_rf   = scaler.inverse_transform(rf.predict(x_test))

rmse_lstm = lstm.evaluate(preds_lstm, y_test)["rmse"]
rmse_rf   = rf.evaluate(preds_rf,   y_test)["rmse"]

print(f"LSTM RMSE : {rmse_lstm:.4f}")
print(f"RF RMSE   : {rmse_rf:.4f}")

In [ ]:
train_slice = df[:train_len]
valid_slice = df[train_len:].copy()
valid_slice["Predictions"]    = preds_lstm[:len(valid_slice)]
valid_slice["RF Predictions"] = preds_rf[:len(valid_slice)]

plot_predictions(train_slice, valid_slice)

## Random Forest — Feature Importances

In [ ]:
importances = rf.feature_importances
lag_labels  = [f"t-{LOOKBACK_WINDOW - i}" for i in range(LOOKBACK_WINDOW)]

plt.figure(figsize=(16, 4))
plt.bar(lag_labels, importances, color="darkorange", edgecolor="white")
plt.xticks(lag_labels[::5], rotation=45, ha="right")
plt.title("RF Feature Importances (60 Lag Days)")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()

## 60-Day Future Forecast

In [ ]:
future_usd, future_dates = build_future_sequence(scaled_data, lstm, scaler)
future_df = pd.DataFrame({"Future Predictions": future_usd.flatten()}, index=future_dates)
future_df.head(10)

In [ ]:
train_full = df[:train_len]
valid_full = df[train_len:].copy()
valid_full["Predictions"]    = preds_lstm[:len(valid_full)]
valid_full["RF Predictions"] = preds_rf[:len(valid_full)]

plot_predictions(train_full, valid_full, future_df)

In [ ]:
# Determine the better model
better = "LSTM" if rmse_lstm < rmse_rf else "Random Forest"

print("=" * 52)
print("          FINAL RESULTS")
print("=" * 52)
print(f"  Ticker          : {PRIMARY_TICKER} ({PRIMARY_TICKER_NAME})")
print(f"  Training data   : {LSTM_START_DATE} → present")
print(f"  Train/Test split: {int(TRAIN_SPLIT_RATIO*100)}% / {int((1-TRAIN_SPLIT_RATIO)*100)}%")
print(f"  Lookback window : {LOOKBACK_WINDOW} days")
print(f"  Epochs (max)    : {EPOCHS}  | Stopped at: {len(lstm.history.history['loss'])}")
print(f"  Batch size      : {BATCH_SIZE}  | Dropout: {DROPOUT_RATE}")
print()
print(f"  {'Model':<22} {'RMSE':>10}")
print("  " + "-" * 33)
print(f"  {'LSTM':<22} {rmse_lstm:>10.4f}")
print(f"  {'Random Forest':<22} {rmse_rf:>10.4f}")
print()
print(f"  Best model: {better}")
print(f"  60-day forecast generated (business days only) ✅")
print("=" * 52)